In [ ]:
import numpy as np
import pandas as pd
from cart import RegressionTree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from typing import List, Tuple, Any


## 初始化提升树模型

按照上面[[#第一步：初始化提升树模型 $f_0(x)$]]的要点，可以用下面这个式子：

$$
\begin{align}
f_0(x) = arg\ min \sum_{i=1}^m \frac{1}{2}(y_{i}-c)^2 \\
\end{align}
$$

对 $c$ 做求导为0，即通过求 $y$ 的均值得到 $c$
$$
\frac{\sum_{i=1}^n y_i}{n} = 0
$$

In [2]:
def compute_initial_prediction(y: np.ndarray) -> float:
    """
    动作 1：计算初始基准预测值 f_0(x) [cite: 53]

    数学推导：在平方损失下，使得初始损失极小化的常数 c 就是真实标签 y 的均值。

    Args:
        y (np.ndarray): 训练集的真实标签，形状为 (n_samples,)

    Returns:
        float: 初始预测常数 f0
    """
    return np.mean(y)

## 计算均值函数的梯度

它的本质就是对损失函数 $L(y, f(x)) = \frac{1}{2}(y - f(x))^2$ 求 $f(x)$ 的偏导。

$$\frac{\partial L}{\partial f(x)} = -(y - f(x))$$

即

$$r_{mi} = -\left[-(y_i - f_0(x_i))\right] = y_i - f_{m-1}(x_i)$$

其中:
- $m$ 为循环的次数。
- $f_{m-1}(x)$ 为前一次计算得到的标签。

下面的代码中，我使用了预测值减去原始值的模式，这是正梯度，刚好和上面公式相反。

In [3]:
def compute_negative_gradient(y: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    动作 2：计算损失函数的负梯度，作为残差的近似值 [cite: 55]

    数学推导：对于平方损失 L = 0.5 * (y - y_pred)^2，其对 y_pred 的负梯度恰好为 y - y_pred。

    Args:
        y (np.ndarray): 真实标签，形状为 (n_samples,)
        y_pred (np.ndarray): 当前模型的累积预测值，形状为 (n_samples,)

    Returns:
        np.ndarray: 计算出的负梯度（残差）数组，形状为 (n_samples,)
    """
    return y_pred - y

由于在计算梯度的时候使用了正梯度，而我们需要梯度下降，所以在更新时用减法（-）去抵消了它。

In [ ]:
def train_gbdt_regressor(
    X: np.ndarray,
    y: np.ndarray,
    n_estimators: int = 100,
    learning_rate: float = 0.1,
    max_depth: int = 3,
    min_samples_split: int = 2,
) -> Tuple[float, List[RegressionTree]]:
    """
    动作 3 & 4：GBDT 核心训练组装流水线

    将初始计算、残差计算和单棵树的训练串联起来。

    Args:
        X (np.ndarray): 训练特征数据，形状为 (n_samples, n_features)
        y (np.ndarray): 训练标签数据，形状为 (n_samples,)
        n_estimators (int): 迭代次数（树的棵数）
        learning_rate (float): 学习率（步长），用于防止过拟合
        max_depth (int): CART 回归树的最大深度
        min_samples_split (int): CART 回归树的结点最小分裂样本数

    Returns:
        Tuple[float, List[RegressionTree]]:
            - float: 初始预测基准值 f0
            - List: 训练好的 CART 回归树集合
    """
    # 1: 调用 compute_initial_prediction 获取 f0
    f0 = compute_initial_prediction(y)
    # 2: 初始化一个全为 f0 的数组 y_pred，作为累积预测的起点
    y_pred = np.full(X.shape[0], f0)
    # 3: 创建一个空列表 trees，准备盛放训练好的树
    trees = []
    # 4: 开启 for 循环，迭代 n_estimators 次
    for i in range(n_estimators):
        # 4.1 调用 compute_negative_gradient 计算当前残差
        y_residuals = compute_negative_gradient(y.flatten(), y_pred)
        # 4.2 实例化一棵 RegressionTree，并用 (X, 残差) 拟合它
        regression_tree = RegressionTree(min_samples_split=min_samples_split, max_depth=max_depth)
        regression_tree.fit(X, y_residuals.reshape(-1, 1))
        # 4.3 获取这棵新树对 X 的预测结果，该预测结果是一个 list
        y_residuals_pred = regression_tree.predict(X)
        # 4.4 更新 y_pred：将新树的预测结果乘以 learning_rate 后累加到 y_pred 上
        y_pred = y_pred - np.array(y_residuals_pred) * learning_rate
        # 4.5 将训练好的这棵树 append 到 trees 列表中
        trees.append(regression_tree)
    # 5: 返回 (f0, trees)
    return f0, trees

In [ ]:
def predict_gbdt_regressor(
    X: np.ndarray,
    f0: float,
    trees: List[RegressionTree],
    learning_rate: float
) -> np.ndarray:
    """
    动作 5：GBDT 最终预测函数 [cite: 66-67]

    将基础预测值与所有树的修正值累加，得到最终预测结果。

    Args:
        X (np.ndarray): 测试集的特征数据
        f0 (float): 训练时得到的初始基准值
        trees (List[RegressionTree]): 训练好的回归树集合
        learning_rate (float): 学习率

    Returns:
        np.ndarray: 最终的预测标签数组
    """
    # 1: 初始化预测数组 y_pred，所有元素值为 f0
    y_pred = np.full(X.shape[0], f0)
    # 2: 遍历 trees 列表中的每一棵树
    for tree in trees:
        # 2.1 获取当前树对 X 的预测结果
        y_tree_pred = tree.predict(X)
        # 2.2 将预测结果乘以 learning_rate 后累加到 y_pred 上
        y_pred = y_pred - np.array(y_tree_pred) * learning_rate
    # 3: 返回 y_pred
    return y_pred

In [6]:
# 波士顿房价数据集的原始 URL
data_url = "http://lib.stat.cmu.edu/datasets/boston"

# 从 URL 加载数据
raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)

# 处理数据
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])  # 拼接特征数据
target = raw_df.values[1::2, 2]  # 目标变量

# 将数据和目标变量转换为 NumPy 数组
X = np.array(data)
y = np.array(target)
y = y.reshape(-1,1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [7]:
print(f'X_train: {X_train.shape}\n' \
      f'y_train: {y_train.shape}\n' \
      f'X_test: {X_test.shape}\n' \
      f'y_test: {y_test.shape}')

X_train: (354, 13)
y_train: (354, 1)
X_test: (152, 13)
y_test: (152, 1)


In [10]:
# 模型训练
f0, trees = train_gbdt_regressor(
    X_train,
    y_train,
    n_estimators=15,
    learning_rate=0.5,
    min_samples_split=4
)

# 模型预测
y_pred = predict_gbdt_regressor(
    X_test,
    f0,
    trees,
    learning_rate=0.5
)
# 计算模型预测的均方误差
mse = mean_squared_error(y_test, y_pred)
print ("Mean Squared Error of NumPy GBRT:", mse)

Mean Squared Error of NumPy GBRT: 13.150644317013496
